In [1]:
import numpy as np
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import mean_squared_error, accuracy_score, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import shap
from xgboost import XGBClassifier
# import random forest regressor
from sklearn.ensemble import RandomForestRegressor
#import linear regression
from sklearn.linear_model import LinearRegression
# import tqdm
from tqdm import tqdm
import tqdm
#import r2_score
from sklearn.metrics import r2_score
#import confusion matrix
from sklearn.metrics import confusion_matrix
# import roc auc score
from sklearn.metrics import roc_auc_score
from ipcch.food_crisis_functions import *
import json
with open("forecasting_hyperparameters.json", "r") as file:
    best_params_xgb_regressor= json.load(file)
    
with open("forecasting_hyperparameters_p3.json", "r") as file:
    best_params_xgb_regressor_for_p3= json.load(file)

# read csv
df_nowcasting = pd.read_csv(r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\nowcasting_subset_IPCCH_v0318_no_lat_lon.csv')
df_forecasting = pd.read_csv(r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\forecasting_subset_IPCCH_v1210.csv')
#rename infra_index_m12 to infra_index_m12_l12, rename infra_index_s12 to infra_index_s12_l12
# add dummys for area_id and month year
#df = pd.concat([df, pd.get_dummies(df['area_id'], prefix='area_id')], axis=1)
#df = pd.concat([df, pd.get_dummies(df['date'], prefix='month_year')], axis=1)
# drop lat and lon
#df = df.drop(['lat', 'lon'], axis=1)
###drop fews_ipc_ha
#df = df.drop(['fews_ipc_ha'], axis=1)
# random split train and test
df_origin = df_forecasting.copy()
df = df_forecasting.copy()
y_pred_test = pd.DataFrame()
model_stats = pd.DataFrame()
#select phase1_percent is not na
df = df_origin[df_origin['phase1_percent'].notna()]
df_nowcasting = df_nowcasting[df_nowcasting['phase1_percent'].notna()]
# Sort by region and date
# prepare date from year and month
df['date'] = pd.to_datetime(df[['year', 'month']].assign(DAY=1))
# check for inf and replace with na
df = df.replace([np.inf, -np.inf], np.nan)
# replace df['overall_phase], 0 as 1, >5 as 5
df['overall_phase'] = df['overall_phase'].apply(lambda x: 1 if x < 1 else (5 if x > 5 else x))

df_nowcasting['date'] = pd.to_datetime(df_nowcasting[['year', 'month']].assign(DAY=1))
df_nowcasting = df_nowcasting.replace([np.inf, -np.inf], np.nan)
df_nowcasting['overall_phase'] = df_nowcasting['overall_phase'].apply(lambda x: 1 if x < 1 else (5 if x > 5 else x))
df = df.sort_values(by=['area_id', 'date'])
df_nowcasting = df_nowcasting.sort_values(by=['area_id', 'date'])
#drop overall phase
df = df.drop(['overall_phase'], axis=1)
#for each region, set last observation to be test set
# create a series of new outcome, phase2_worse=phase2_percent+phase3_percent+phase4_percent+phase5_percent, phase3_worse=phase3_percent+phase4_percent+phase5_percent, phase4_worse=phase4_percent+phase5_percent, phase5_worse=phase5_percent
df['phase2_worse'] = df['phase2_percent'] + df['phase3_percent'] + df['phase4_percent'] + df['phase5_percent']
df['phase3_worse'] = df['phase3_percent'] + df['phase4_percent'] + df['phase5_percent']
df['phase4_worse'] = df['phase4_percent'] + df['phase5_percent']
df['phase5_worse'] = df['phase5_percent']
#drop phase2_percent, phase3_percent, phase4_percent, phase5_percent, phase1_percent
df = df.drop(['phase2_percent', 'phase3_percent', 'phase4_percent', 'phase5_percent', 'phase1_percent'], axis=1)
# Splitting the data
#test_df = df.groupby('area_id').tail(1)
#train_df = df.drop(test_df.index)
#test_df = test_df.drop(['area_id','date'], axis=1)
#train_df = train_df.drop(['area_id','date'], axis=1)
y_pred_test = pd.DataFrame()
# drop anything after 2022-01-01
#df = df[df['date'] < '2021-01-01']
shape_values_df_ensemble = pd.DataFrame()
df_result = pd.DataFrame()
date = "2024-01-01"  # Define the 'date' variable
#order unique_dates
y_pred_test=pd.DataFrame()

diff_set = ['CC',
 'CC_m12',
 'CPI',
 'CPI_m12',
 'EVI_mean',
 'EVI_mean_m12',
 'EVI_std',
 'EVI_std_m12',
 'Evap_tavg_mean',
 'Evap_tavg_mean_m12',
 'Evap_tavg_stdDev',
 'Evap_tavg_stdDev_m12',
 'FAO_price',
 'FAO_price_m12',
 'GDP',
 'GDP_m12',
 'GPP_mean',
 'GPP_mean_m12',
 'GPP_std',
 'GPP_std_m12',
 'LWdown_f_tavg_mean',
 'LWdown_f_tavg_mean_m12',
 'LWdown_f_tavg_stdDev',
 'LWdown_f_tavg_stdDev_m12',
 'Lwnet_tavg_mean',
 'Lwnet_tavg_mean_m12',
 'Lwnet_tavg_stdDev',
 'Lwnet_tavg_stdDev_m12',
 'Psurf_f_tavg_mean',
 'Psurf_f_tavg_mean_m12',
 'Psurf_f_tavg_stdDev',
 'Psurf_f_tavg_stdDev_m12',
 'Qair_f_tavg_mean',
 'Qair_f_tavg_mean_m12',
 'Qair_f_tavg_stdDev',
 'Qair_f_tavg_stdDev_m12',
 'Qg_tavg_mean',
 'Qg_tavg_mean_m12',
 'Qg_tavg_stdDev',
 'Qg_tavg_stdDev_m12',
 'Qh_tavg_mean',
 'Qh_tavg_mean_m12',
 'Qh_tavg_stdDev',
 'Qh_tavg_stdDev_m12',
 'Qle_tavg_mean',
 'Qle_tavg_mean_m12',
 'Qle_tavg_stdDev',
 'Qle_tavg_stdDev_m12',
 'Qs_tavg_mean',
 'Qs_tavg_mean_m12',
 'Qs_tavg_stdDev',
 'Qs_tavg_stdDev_m12',
 'Qsb_tavg_mean',
 'Qsb_tavg_mean_m12',
 'Qsb_tavg_stdDev',
 'Qsb_tavg_stdDev_m12',
 'RadT_tavg_mean',
 'RadT_tavg_mean_m12',
 'RadT_tavg_stdDev',
 'RadT_tavg_stdDev_m12',
 'Rainf_f_tavg_mean',
 'Rainf_f_tavg_mean_m12',
 'Rainf_f_tavg_stdDev',
 'Rainf_f_tavg_stdDev_m12',
 'SWE_inst_mean',
 'SWE_inst_mean_m12',
 'SWE_inst_stdDev',
 'SWE_inst_stdDev_m12',
 'SWdown_f_tavg_mean',
 'SWdown_f_tavg_mean_m12',
 'SWdown_f_tavg_stdDev',
 'SWdown_f_tavg_stdDev_m12',
 'SnowCover_inst_mean',
 'SnowCover_inst_mean_m12',
 'SnowCover_inst_stdDev',
 'SnowCover_inst_stdDev_m12',
 'SnowDepth_inst_mean',
 'SnowDepth_inst_mean_m12',
 'SnowDepth_inst_stdDev',
 'SnowDepth_inst_stdDev_m12',
 'Snowf_tavg_mean',
 'Snowf_tavg_mean_m12',
 'Snowf_tavg_stdDev',
 'Snowf_tavg_stdDev_m12',
 'SoilMoi00_10cm_tavg_mean',
 'SoilMoi00_10cm_tavg_mean_m12',
 'SoilMoi00_10cm_tavg_stdDev',
 'SoilMoi00_10cm_tavg_stdDev_m12',
 'SoilMoi100_200cm_tavg_mean',
 'SoilMoi100_200cm_tavg_mean_m12',
 'SoilMoi100_200cm_tavg_stdDev',
 'SoilMoi100_200cm_tavg_stdDev_m12',
 'SoilMoi10_40cm_tavg_mean',
 'SoilMoi10_40cm_tavg_mean_m12',
 'SoilMoi10_40cm_tavg_stdDev',
 'SoilMoi10_40cm_tavg_stdDev_m12',
 'SoilMoi40_100cm_tavg_mean',
 'SoilMoi40_100cm_tavg_mean_m12',
 'SoilMoi40_100cm_tavg_stdDev',
 'SoilMoi40_100cm_tavg_stdDev_m12',
 'SoilTemp00_10cm_tavg_mean',
 'SoilTemp00_10cm_tavg_mean_m12',
 'SoilTemp00_10cm_tavg_stdDev',
 'SoilTemp00_10cm_tavg_stdDev_m12',
 'SoilTemp100_200cm_tavg_mean',
 'SoilTemp100_200cm_tavg_mean_m12',
 'SoilTemp100_200cm_tavg_stdDev',
 'SoilTemp100_200cm_tavg_stdDev_m12',
 'SoilTemp10_40cm_tavg_mean',
 'SoilTemp10_40cm_tavg_mean_m12',
 'SoilTemp10_40cm_tavg_stdDev',
 'SoilTemp10_40cm_tavg_stdDev_m12',
 'SoilTemp40_100cm_tavg_mean',
 'SoilTemp40_100cm_tavg_mean_m12',
 'SoilTemp40_100cm_tavg_stdDev',
 'SoilTemp40_100cm_tavg_stdDev_m12',
 'Swnet_tavg_mean',
 'Swnet_tavg_mean_m12',
 'Swnet_tavg_stdDev',
 'Swnet_tavg_stdDev_m12',
 'Tair_f_tavg_mean',
 'Tair_f_tavg_mean_m12',
 'Tair_f_tavg_stdDev',
 'Tair_f_tavg_stdDev_m12',
 'WFP_Price',
 'WFP_Price_m12',
 'WFP_Price_std',
 'WFP_Price_std_m12',
 'Wind_f_tavg_mean',
 'Wind_f_tavg_mean_m12',
 'Wind_f_tavg_stdDev',
 'Wind_f_tavg_stdDev_m12',
 'event_count_battles',
 'event_count_battles_m12',
 'event_count_battles_w10',
 'event_count_battles_w10_m12',
 'event_count_battles_w5',
 'event_count_battles_w5_m12',
 'event_count_explosions',
 'event_count_explosions_m12',
 'event_count_explosions_w10',
 'event_count_explosions_w10_m12',
 'event_count_explosions_w5',
 'event_count_explosions_w5_m12',
 'event_count_violence',
 'event_count_violence_m12',
 'event_count_violence_w10',
 'event_count_violence_w10_m12',
 'event_count_violence_w5',
 'event_count_violence_w5_m12',
 'gini',
 'gini_m12',
 'nightlight_mean',
 'nightlight_mean_m12',
 'nightlight_std',
 'nightlight_std_m12',
 'sum_fatalities_battles',
 'sum_fatalities_battles_m12',
 'sum_fatalities_battles_w10',
 'sum_fatalities_battles_w10_m12',
 'sum_fatalities_battles_w5',
 'sum_fatalities_battles_w5_m12',
 'sum_fatalities_explosions',
 'sum_fatalities_explosions_w10',
 'sum_fatalities_explosions_w10_m12',
 'sum_fatalities_explosions_w5',
 'sum_fatalities_explosions_w5_m12',
 'sum_fatalities_violence',
 'sum_fatalities_violence_m12',
 'sum_fatalities_violence_w10',
 'sum_fatalities_violence_w10_m12',
 'sum_fatalities_violence_w5',
 'sum_fatalities_violence_w5_m12']


# --- Pre-merge nowcasting diff_set features into forecasting DataFrame ---
now_cols_to_merge = [c for c in diff_set if c in df_nowcasting.columns]
nowcasting_merge = df_nowcasting[['area_id', 'date'] + now_cols_to_merge].copy()
nowcasting_merge = nowcasting_merge.rename(columns={c: f'now_{c}' for c in now_cols_to_merge})
now_diff_set = [f'now_{c}' for c in now_cols_to_merge]
df = df.merge(nowcasting_merge, on=['area_id', 'date'], how='left')
print(f"Merged {len(now_diff_set)} nowcasting features. df shape: {df.shape}")


Merged 173 nowcasting features. df shape: (29621, 397)


In [2]:
y_pred_test = pd.DataFrame()
date = "2024-01-01"

for i in range(2, 6):
    train_df = df[df['date'] < date].copy()
    test_df = df[df['date'] >= date].copy()

    # Extract nowcasting features before dropping columns
    train_now_feats = train_df[now_diff_set]
    test_now_feats = test_df[now_diff_set]

    # Drop date, area_id, and now_* columns for Layer 1
    train_df = train_df.drop(['date', 'area_id'] + now_diff_set, axis=1)
    test_df = test_df.drop(['date', 'area_id'] + now_diff_set, axis=1)

    # Phase filtering
    train_df_new = train_df.drop([f'phase{j}_worse' for j in range(2, 6) if j != i], axis=1)
    test_df_new = test_df.drop([f'phase{j}_worse' for j in range(2, 6) if j != i], axis=1)
    train_df_new = train_df_new.dropna(subset=[f'phase{i}_worse'])
    test_df_new = test_df_new.dropna(subset=[f'phase{i}_worse'])
    test_index = test_df_new.index

    # LAYER 1
    X_train_L1 = train_df_new.drop(f'phase{i}_worse', axis=1)
    y_train = train_df_new[f'phase{i}_worse']
    X_test_L1 = test_df_new.drop(f'phase{i}_worse', axis=1)
    y_test = test_df_new[f'phase{i}_worse']

    layer_params = best_params_xgb_regressor_for_p3 if i == 3 else best_params_xgb_regressor
    model_L1 = xgb.XGBRegressor(**layer_params)
    model_L1.fit(X_train_L1, y_train)

    y_pred_L1 = pd.Series(model_L1.predict(X_test_L1), index=X_test_L1.index)
    train_residuals = pd.Series(
        y_train.to_numpy() - model_L1.predict(X_train_L1),
        index=X_train_L1.index
    )

    # LAYER 2 (index-aligned from same DataFrame)
    X_train_L2 = train_now_feats.loc[X_train_L1.index]
    X_test_L2 = test_now_feats.loc[X_test_L1.index]

    # Drop rows where ALL nowcasting features are NaN (XGBoost handles partial NaN natively)
    train_mask = X_train_L2.notna().any(axis=1)
    test_mask = X_test_L2.notna().any(axis=1)
    X_train_L2 = X_train_L2[train_mask]
    X_test_L2 = X_test_L2[test_mask]
    train_residuals_aligned = train_residuals.loc[X_train_L2.index]
    y_pred_L1_aligned = y_pred_L1.loc[X_test_L2.index]
    y_test_aligned = y_test.loc[X_test_L2.index]

    if X_train_L2.empty or X_test_L2.empty:
        print(f"Phase {i}: skipped (empty Layer 2 data)")
        continue

    model_L2 = xgb.XGBRegressor(**layer_params)
    model_L2.fit(X_train_L2, train_residuals_aligned)
    residual_pred = model_L2.predict(X_test_L2)
    y_pred_final = y_pred_L1_aligned.to_numpy() + residual_pred

    # Store in reference format
    row = {'y_pred': y_pred_final, 'y_test': y_test_aligned.to_numpy(), 'phase': [i]*len(y_pred_final)}
    if i == 5:
        row['test_index'] = X_test_L2.index
    y_pred_test = pd.concat([y_pred_test, pd.DataFrame(row)], ignore_index=True)

y_pred_test = convert_prob_to_phase(y_pred_test)
y_test = y_pred_test['overall_phase']
y_pred = y_pred_test['overall_phase_pred']
cm = confusion_matrix(y_test, y_pred)
accuracy_score_new, sensitivity, precision, overall_r2 = all_metrics(y_test, y_pred, cm, y_pred_test)
print("2024")
print("Accuracy:", accuracy_score_new)
print("Sensitivity:", sensitivity)
print("Precision:", precision)
print("Overall R²(Phase 3 or more):", overall_r2)


2024
Accuracy: 0.6719441210710128
Sensitivity: 0.8314113124656782
Precision: 0.7285851780558229
Overall R²(Phase 3 or more): 0.47926656877458207


In [3]:
import os
out_dir = 'results/predictions/nowcasting'
os.makedirs(out_dir, exist_ok=True)

y_pred_test = y_pred_test.loc[:, (y_pred_test != 0).any(axis=0)]
df_extracted = df[['area_id', 'date']].copy()
df_extracted['test_index'] = df_extracted.index
y_pred_test = y_pred_test.merge(df_extracted, on='test_index', how='left')

PATH = r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\IPCCH_2017_2025_final_v12102025_with_zscores.csv'
df_ipcch = pd.read_csv(PATH)
df_ipcch = df_ipcch[['admin_code', 'lat', 'lon']].rename(columns={'admin_code': 'area_id'})
y_pred_test = y_pred_test.merge(df_ipcch, on='area_id', how='left')
y_pred_test.to_csv(os.path.join(out_dir, 'nowcasting_y_pred_test_2024.csv'), index=False)
%reset -f


In [4]:
import numpy as np
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import mean_squared_error, accuracy_score, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import shap
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from tqdm import tqdm
import tqdm
from sklearn.metrics import r2_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score
from ipcch.food_crisis_functions import *
import json
with open("forecasting_hyperparameters.json", "r") as file:
    best_params_xgb_regressor= json.load(file)

with open("forecasting_hyperparameters_p3.json", "r") as file:
    best_params_xgb_regressor_for_p3= json.load(file)

df_nowcasting = pd.read_csv(r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\nowcasting_subset_IPCCH_v0318_no_lat_lon.csv')
df_forecasting = pd.read_csv(r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\forecasting_subset_IPCCH_v1210.csv')
df_origin = df_forecasting.copy()
df = df_forecasting.copy()
df = df_origin[df_origin['phase1_percent'].notna()]
df_nowcasting = df_nowcasting[df_nowcasting['phase1_percent'].notna()]
df['date'] = pd.to_datetime(df[['year', 'month']].assign(DAY=1))
df = df.replace([np.inf, -np.inf], np.nan)
df['overall_phase'] = df['overall_phase'].apply(lambda x: 1 if x < 1 else (5 if x > 5 else x))
df_nowcasting['date'] = pd.to_datetime(df_nowcasting[['year', 'month']].assign(DAY=1))
df_nowcasting = df_nowcasting.replace([np.inf, -np.inf], np.nan)
df_nowcasting['overall_phase'] = df_nowcasting['overall_phase'].apply(lambda x: 1 if x < 1 else (5 if x > 5 else x))
df = df.sort_values(by=['area_id', 'date'])
df_nowcasting = df_nowcasting.sort_values(by=['area_id', 'date'])
df = df.drop(['overall_phase'], axis=1)
df['phase2_worse'] = df['phase2_percent'] + df['phase3_percent'] + df['phase4_percent'] + df['phase5_percent']
df['phase3_worse'] = df['phase3_percent'] + df['phase4_percent'] + df['phase5_percent']
df['phase4_worse'] = df['phase4_percent'] + df['phase5_percent']
df['phase5_worse'] = df['phase5_percent']
df = df.drop(['phase2_percent', 'phase3_percent', 'phase4_percent', 'phase5_percent', 'phase1_percent'], axis=1)

diff_set = ['CC',
 'CC_m12',
 'CPI',
 'CPI_m12',
 'EVI_mean',
 'EVI_mean_m12',
 'EVI_std',
 'EVI_std_m12',
 'Evap_tavg_mean',
 'Evap_tavg_mean_m12',
 'Evap_tavg_stdDev',
 'Evap_tavg_stdDev_m12',
 'FAO_price',
 'FAO_price_m12',
 'GDP',
 'GDP_m12',
 'GPP_mean',
 'GPP_mean_m12',
 'GPP_std',
 'GPP_std_m12',
 'LWdown_f_tavg_mean',
 'LWdown_f_tavg_mean_m12',
 'LWdown_f_tavg_stdDev',
 'LWdown_f_tavg_stdDev_m12',
 'Lwnet_tavg_mean',
 'Lwnet_tavg_mean_m12',
 'Lwnet_tavg_stdDev',
 'Lwnet_tavg_stdDev_m12',
 'Psurf_f_tavg_mean',
 'Psurf_f_tavg_mean_m12',
 'Psurf_f_tavg_stdDev',
 'Psurf_f_tavg_stdDev_m12',
 'Qair_f_tavg_mean',
 'Qair_f_tavg_mean_m12',
 'Qair_f_tavg_stdDev',
 'Qair_f_tavg_stdDev_m12',
 'Qg_tavg_mean',
 'Qg_tavg_mean_m12',
 'Qg_tavg_stdDev',
 'Qg_tavg_stdDev_m12',
 'Qh_tavg_mean',
 'Qh_tavg_mean_m12',
 'Qh_tavg_stdDev',
 'Qh_tavg_stdDev_m12',
 'Qle_tavg_mean',
 'Qle_tavg_mean_m12',
 'Qle_tavg_stdDev',
 'Qle_tavg_stdDev_m12',
 'Qs_tavg_mean',
 'Qs_tavg_mean_m12',
 'Qs_tavg_stdDev',
 'Qs_tavg_stdDev_m12',
 'Qsb_tavg_mean',
 'Qsb_tavg_mean_m12',
 'Qsb_tavg_stdDev',
 'Qsb_tavg_stdDev_m12',
 'RadT_tavg_mean',
 'RadT_tavg_mean_m12',
 'RadT_tavg_stdDev',
 'RadT_tavg_stdDev_m12',
 'Rainf_f_tavg_mean',
 'Rainf_f_tavg_mean_m12',
 'Rainf_f_tavg_stdDev',
 'Rainf_f_tavg_stdDev_m12',
 'SWE_inst_mean',
 'SWE_inst_mean_m12',
 'SWE_inst_stdDev',
 'SWE_inst_stdDev_m12',
 'SWdown_f_tavg_mean',
 'SWdown_f_tavg_mean_m12',
 'SWdown_f_tavg_stdDev',
 'SWdown_f_tavg_stdDev_m12',
 'SnowCover_inst_mean',
 'SnowCover_inst_mean_m12',
 'SnowCover_inst_stdDev',
 'SnowCover_inst_stdDev_m12',
 'SnowDepth_inst_mean',
 'SnowDepth_inst_mean_m12',
 'SnowDepth_inst_stdDev',
 'SnowDepth_inst_stdDev_m12',
 'Snowf_tavg_mean',
 'Snowf_tavg_mean_m12',
 'Snowf_tavg_stdDev',
 'Snowf_tavg_stdDev_m12',
 'SoilMoi00_10cm_tavg_mean',
 'SoilMoi00_10cm_tavg_mean_m12',
 'SoilMoi00_10cm_tavg_stdDev',
 'SoilMoi00_10cm_tavg_stdDev_m12',
 'SoilMoi100_200cm_tavg_mean',
 'SoilMoi100_200cm_tavg_mean_m12',
 'SoilMoi100_200cm_tavg_stdDev',
 'SoilMoi100_200cm_tavg_stdDev_m12',
 'SoilMoi10_40cm_tavg_mean',
 'SoilMoi10_40cm_tavg_mean_m12',
 'SoilMoi10_40cm_tavg_stdDev',
 'SoilMoi10_40cm_tavg_stdDev_m12',
 'SoilMoi40_100cm_tavg_mean',
 'SoilMoi40_100cm_tavg_mean_m12',
 'SoilMoi40_100cm_tavg_stdDev',
 'SoilMoi40_100cm_tavg_stdDev_m12',
 'SoilTemp00_10cm_tavg_mean',
 'SoilTemp00_10cm_tavg_mean_m12',
 'SoilTemp00_10cm_tavg_stdDev',
 'SoilTemp00_10cm_tavg_stdDev_m12',
 'SoilTemp100_200cm_tavg_mean',
 'SoilTemp100_200cm_tavg_mean_m12',
 'SoilTemp100_200cm_tavg_stdDev',
 'SoilTemp100_200cm_tavg_stdDev_m12',
 'SoilTemp10_40cm_tavg_mean',
 'SoilTemp10_40cm_tavg_mean_m12',
 'SoilTemp10_40cm_tavg_stdDev',
 'SoilTemp10_40cm_tavg_stdDev_m12',
 'SoilTemp40_100cm_tavg_mean',
 'SoilTemp40_100cm_tavg_mean_m12',
 'SoilTemp40_100cm_tavg_stdDev',
 'SoilTemp40_100cm_tavg_stdDev_m12',
 'Swnet_tavg_mean',
 'Swnet_tavg_mean_m12',
 'Swnet_tavg_stdDev',
 'Swnet_tavg_stdDev_m12',
 'Tair_f_tavg_mean',
 'Tair_f_tavg_mean_m12',
 'Tair_f_tavg_stdDev',
 'Tair_f_tavg_stdDev_m12',
 'WFP_Price',
 'WFP_Price_m12',
 'WFP_Price_std',
 'WFP_Price_std_m12',
 'Wind_f_tavg_mean',
 'Wind_f_tavg_mean_m12',
 'Wind_f_tavg_stdDev',
 'Wind_f_tavg_stdDev_m12',
 'event_count_battles',
 'event_count_battles_m12',
 'event_count_battles_w10',
 'event_count_battles_w10_m12',
 'event_count_battles_w5',
 'event_count_battles_w5_m12',
 'event_count_explosions',
 'event_count_explosions_m12',
 'event_count_explosions_w10',
 'event_count_explosions_w10_m12',
 'event_count_explosions_w5',
 'event_count_explosions_w5_m12',
 'event_count_violence',
 'event_count_violence_m12',
 'event_count_violence_w10',
 'event_count_violence_w10_m12',
 'event_count_violence_w5',
 'event_count_violence_w5_m12',
 'gini',
 'gini_m12',
 'nightlight_mean',
 'nightlight_mean_m12',
 'nightlight_std',
 'nightlight_std_m12',
 'sum_fatalities_battles',
 'sum_fatalities_battles_m12',
 'sum_fatalities_battles_w10',
 'sum_fatalities_battles_w10_m12',
 'sum_fatalities_battles_w5',
 'sum_fatalities_battles_w5_m12',
 'sum_fatalities_explosions',
 'sum_fatalities_explosions_w10',
 'sum_fatalities_explosions_w10_m12',
 'sum_fatalities_explosions_w5',
 'sum_fatalities_explosions_w5_m12',
 'sum_fatalities_violence',
 'sum_fatalities_violence_m12',
 'sum_fatalities_violence_w10',
 'sum_fatalities_violence_w10_m12',
 'sum_fatalities_violence_w5',
 'sum_fatalities_violence_w5_m12']

# --- Pre-merge nowcasting diff_set features into forecasting DataFrame ---
now_cols_to_merge = [c for c in diff_set if c in df_nowcasting.columns]
nowcasting_merge = df_nowcasting[['area_id', 'date'] + now_cols_to_merge].copy()
nowcasting_merge = nowcasting_merge.rename(columns={c: f'now_{c}' for c in now_cols_to_merge})
now_diff_set = [f'now_{c}' for c in now_cols_to_merge]
df = df.merge(nowcasting_merge, on=['area_id', 'date'], how='left')
print(f"Merged {len(now_diff_set)} nowcasting features. df shape: {df.shape}")

y_pred_test = pd.DataFrame()
date = "2023-01-01"
cutoff_date = "2024-01-01"

for i in range(2, 6):
    train_df = df[df['date'] < date].copy()
    test_df = df[(df['date'] >= date) & (df['date'] < cutoff_date)].copy()

    # Extract nowcasting features before dropping columns
    train_now_feats = train_df[now_diff_set]
    test_now_feats = test_df[now_diff_set]

    # Drop date, area_id, and now_* columns for Layer 1
    train_df = train_df.drop(['date', 'area_id'] + now_diff_set, axis=1)
    test_df = test_df.drop(['date', 'area_id'] + now_diff_set, axis=1)

    # Phase filtering
    train_df_new = train_df.drop([f'phase{j}_worse' for j in range(2, 6) if j != i], axis=1)
    test_df_new = test_df.drop([f'phase{j}_worse' for j in range(2, 6) if j != i], axis=1)
    train_df_new = train_df_new.dropna(subset=[f'phase{i}_worse'])
    test_df_new = test_df_new.dropna(subset=[f'phase{i}_worse'])
    test_index = test_df_new.index

    # LAYER 1
    X_train_L1 = train_df_new.drop(f'phase{i}_worse', axis=1)
    y_train = train_df_new[f'phase{i}_worse']
    X_test_L1 = test_df_new.drop(f'phase{i}_worse', axis=1)
    y_test = test_df_new[f'phase{i}_worse']

    layer_params = best_params_xgb_regressor_for_p3 if i == 3 else best_params_xgb_regressor
    model_L1 = xgb.XGBRegressor(**layer_params)
    model_L1.fit(X_train_L1, y_train)

    y_pred_L1 = pd.Series(model_L1.predict(X_test_L1), index=X_test_L1.index)
    train_residuals = pd.Series(
        y_train.to_numpy() - model_L1.predict(X_train_L1),
        index=X_train_L1.index
    )

    # LAYER 2 (index-aligned from same DataFrame)
    X_train_L2 = train_now_feats.loc[X_train_L1.index]
    X_test_L2 = test_now_feats.loc[X_test_L1.index]

    # Drop rows where ALL nowcasting features are NaN (XGBoost handles partial NaN natively)
    train_mask = X_train_L2.notna().any(axis=1)
    test_mask = X_test_L2.notna().any(axis=1)
    X_train_L2 = X_train_L2[train_mask]
    X_test_L2 = X_test_L2[test_mask]
    train_residuals_aligned = train_residuals.loc[X_train_L2.index]
    y_pred_L1_aligned = y_pred_L1.loc[X_test_L2.index]
    y_test_aligned = y_test.loc[X_test_L2.index]

    if X_train_L2.empty or X_test_L2.empty:
        print(f"Phase {i}: skipped (empty Layer 2 data)")
        continue

    model_L2 = xgb.XGBRegressor(**layer_params)
    model_L2.fit(X_train_L2, train_residuals_aligned)
    residual_pred = model_L2.predict(X_test_L2)
    y_pred_final = y_pred_L1_aligned.to_numpy() + residual_pred

    # Store in reference format
    row = {'y_pred': y_pred_final, 'y_test': y_test_aligned.to_numpy(), 'phase': [i]*len(y_pred_final)}
    if i == 5:
        row['test_index'] = X_test_L2.index
    y_pred_test = pd.concat([y_pred_test, pd.DataFrame(row)], ignore_index=True)

y_pred_test = convert_prob_to_phase(y_pred_test)
y_test = y_pred_test['overall_phase']
y_pred = y_pred_test['overall_phase_pred']
cm = confusion_matrix(y_test, y_pred)
accuracy_score_new, sensitivity, precision, overall_r2 = all_metrics(y_test, y_pred, cm, y_pred_test)
print("2023")
print("Accuracy:", accuracy_score_new)
print("Sensitivity:", sensitivity)
print("Precision:", precision)
print("Overall R²(Phase 3 or more):", overall_r2)


Merged 173 nowcasting features. df shape: (29621, 397)
2023
Accuracy: 0.608730964467005
Sensitivity: 0.8149587178241865
Precision: 0.7561964849031095
Overall R²(Phase 3 or more): 0.5433969365011232


In [5]:
import os
out_dir = 'results/predictions/nowcasting'
os.makedirs(out_dir, exist_ok=True)

y_pred_test = y_pred_test.loc[:, (y_pred_test != 0).any(axis=0)]
df_extracted = df[['area_id', 'date']].copy()
df_extracted['test_index'] = df_extracted.index
y_pred_test = y_pred_test.merge(df_extracted, on='test_index', how='left')

PATH = r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\IPCCH_2017_2025_final_v12102025_with_zscores.csv'
df_ipcch = pd.read_csv(PATH)
df_ipcch = df_ipcch[['admin_code', 'lat', 'lon']].rename(columns={'admin_code': 'area_id'})
y_pred_test = y_pred_test.merge(df_ipcch, on='area_id', how='left')
y_pred_test.to_csv(os.path.join(out_dir, 'nowcasting_y_pred_test_2023.csv'), index=False)
%reset -f


In [6]:
import numpy as np
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import mean_squared_error, accuracy_score, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import shap
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from tqdm import tqdm
import tqdm
from sklearn.metrics import r2_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score
from ipcch.food_crisis_functions import *
import json
with open("forecasting_hyperparameters.json", "r") as file:
    best_params_xgb_regressor= json.load(file)

with open("forecasting_hyperparameters_p3.json", "r") as file:
    best_params_xgb_regressor_for_p3= json.load(file)

df_nowcasting = pd.read_csv(r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\nowcasting_subset_IPCCH_v0318_no_lat_lon.csv')
df_forecasting = pd.read_csv(r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\forecasting_subset_IPCCH_v1210.csv')
df_origin = df_forecasting.copy()
df = df_forecasting.copy()
df = df_origin[df_origin['phase1_percent'].notna()]
df_nowcasting = df_nowcasting[df_nowcasting['phase1_percent'].notna()]
df['date'] = pd.to_datetime(df[['year', 'month']].assign(DAY=1))
df = df.replace([np.inf, -np.inf], np.nan)
df['overall_phase'] = df['overall_phase'].apply(lambda x: 1 if x < 1 else (5 if x > 5 else x))
df_nowcasting['date'] = pd.to_datetime(df_nowcasting[['year', 'month']].assign(DAY=1))
df_nowcasting = df_nowcasting.replace([np.inf, -np.inf], np.nan)
df_nowcasting['overall_phase'] = df_nowcasting['overall_phase'].apply(lambda x: 1 if x < 1 else (5 if x > 5 else x))
df = df.sort_values(by=['area_id', 'date'])
df_nowcasting = df_nowcasting.sort_values(by=['area_id', 'date'])
df = df.drop(['overall_phase'], axis=1)
df['phase2_worse'] = df['phase2_percent'] + df['phase3_percent'] + df['phase4_percent'] + df['phase5_percent']
df['phase3_worse'] = df['phase3_percent'] + df['phase4_percent'] + df['phase5_percent']
df['phase4_worse'] = df['phase4_percent'] + df['phase5_percent']
df['phase5_worse'] = df['phase5_percent']
df = df.drop(['phase2_percent', 'phase3_percent', 'phase4_percent', 'phase5_percent', 'phase1_percent'], axis=1)

diff_set = ['CC',
 'CC_m12',
 'CPI',
 'CPI_m12',
 'EVI_mean',
 'EVI_mean_m12',
 'EVI_std',
 'EVI_std_m12',
 'Evap_tavg_mean',
 'Evap_tavg_mean_m12',
 'Evap_tavg_stdDev',
 'Evap_tavg_stdDev_m12',
 'FAO_price',
 'FAO_price_m12',
 'GDP',
 'GDP_m12',
 'GPP_mean',
 'GPP_mean_m12',
 'GPP_std',
 'GPP_std_m12',
 'LWdown_f_tavg_mean',
 'LWdown_f_tavg_mean_m12',
 'LWdown_f_tavg_stdDev',
 'LWdown_f_tavg_stdDev_m12',
 'Lwnet_tavg_mean',
 'Lwnet_tavg_mean_m12',
 'Lwnet_tavg_stdDev',
 'Lwnet_tavg_stdDev_m12',
 'Psurf_f_tavg_mean',
 'Psurf_f_tavg_mean_m12',
 'Psurf_f_tavg_stdDev',
 'Psurf_f_tavg_stdDev_m12',
 'Qair_f_tavg_mean',
 'Qair_f_tavg_mean_m12',
 'Qair_f_tavg_stdDev',
 'Qair_f_tavg_stdDev_m12',
 'Qg_tavg_mean',
 'Qg_tavg_mean_m12',
 'Qg_tavg_stdDev',
 'Qg_tavg_stdDev_m12',
 'Qh_tavg_mean',
 'Qh_tavg_mean_m12',
 'Qh_tavg_stdDev',
 'Qh_tavg_stdDev_m12',
 'Qle_tavg_mean',
 'Qle_tavg_mean_m12',
 'Qle_tavg_stdDev',
 'Qle_tavg_stdDev_m12',
 'Qs_tavg_mean',
 'Qs_tavg_mean_m12',
 'Qs_tavg_stdDev',
 'Qs_tavg_stdDev_m12',
 'Qsb_tavg_mean',
 'Qsb_tavg_mean_m12',
 'Qsb_tavg_stdDev',
 'Qsb_tavg_stdDev_m12',
 'RadT_tavg_mean',
 'RadT_tavg_mean_m12',
 'RadT_tavg_stdDev',
 'RadT_tavg_stdDev_m12',
 'Rainf_f_tavg_mean',
 'Rainf_f_tavg_mean_m12',
 'Rainf_f_tavg_stdDev',
 'Rainf_f_tavg_stdDev_m12',
 'SWE_inst_mean',
 'SWE_inst_mean_m12',
 'SWE_inst_stdDev',
 'SWE_inst_stdDev_m12',
 'SWdown_f_tavg_mean',
 'SWdown_f_tavg_mean_m12',
 'SWdown_f_tavg_stdDev',
 'SWdown_f_tavg_stdDev_m12',
 'SnowCover_inst_mean',
 'SnowCover_inst_mean_m12',
 'SnowCover_inst_stdDev',
 'SnowCover_inst_stdDev_m12',
 'SnowDepth_inst_mean',
 'SnowDepth_inst_mean_m12',
 'SnowDepth_inst_stdDev',
 'SnowDepth_inst_stdDev_m12',
 'Snowf_tavg_mean',
 'Snowf_tavg_mean_m12',
 'Snowf_tavg_stdDev',
 'Snowf_tavg_stdDev_m12',
 'SoilMoi00_10cm_tavg_mean',
 'SoilMoi00_10cm_tavg_mean_m12',
 'SoilMoi00_10cm_tavg_stdDev',
 'SoilMoi00_10cm_tavg_stdDev_m12',
 'SoilMoi100_200cm_tavg_mean',
 'SoilMoi100_200cm_tavg_mean_m12',
 'SoilMoi100_200cm_tavg_stdDev',
 'SoilMoi100_200cm_tavg_stdDev_m12',
 'SoilMoi10_40cm_tavg_mean',
 'SoilMoi10_40cm_tavg_mean_m12',
 'SoilMoi10_40cm_tavg_stdDev',
 'SoilMoi10_40cm_tavg_stdDev_m12',
 'SoilMoi40_100cm_tavg_mean',
 'SoilMoi40_100cm_tavg_mean_m12',
 'SoilMoi40_100cm_tavg_stdDev',
 'SoilMoi40_100cm_tavg_stdDev_m12',
 'SoilTemp00_10cm_tavg_mean',
 'SoilTemp00_10cm_tavg_mean_m12',
 'SoilTemp00_10cm_tavg_stdDev',
 'SoilTemp00_10cm_tavg_stdDev_m12',
 'SoilTemp100_200cm_tavg_mean',
 'SoilTemp100_200cm_tavg_mean_m12',
 'SoilTemp100_200cm_tavg_stdDev',
 'SoilTemp100_200cm_tavg_stdDev_m12',
 'SoilTemp10_40cm_tavg_mean',
 'SoilTemp10_40cm_tavg_mean_m12',
 'SoilTemp10_40cm_tavg_stdDev',
 'SoilTemp10_40cm_tavg_stdDev_m12',
 'SoilTemp40_100cm_tavg_mean',
 'SoilTemp40_100cm_tavg_mean_m12',
 'SoilTemp40_100cm_tavg_stdDev',
 'SoilTemp40_100cm_tavg_stdDev_m12',
 'Swnet_tavg_mean',
 'Swnet_tavg_mean_m12',
 'Swnet_tavg_stdDev',
 'Swnet_tavg_stdDev_m12',
 'Tair_f_tavg_mean',
 'Tair_f_tavg_mean_m12',
 'Tair_f_tavg_stdDev',
 'Tair_f_tavg_stdDev_m12',
 'WFP_Price',
 'WFP_Price_m12',
 'WFP_Price_std',
 'WFP_Price_std_m12',
 'Wind_f_tavg_mean',
 'Wind_f_tavg_mean_m12',
 'Wind_f_tavg_stdDev',
 'Wind_f_tavg_stdDev_m12',
 'event_count_battles',
 'event_count_battles_m12',
 'event_count_battles_w10',
 'event_count_battles_w10_m12',
 'event_count_battles_w5',
 'event_count_battles_w5_m12',
 'event_count_explosions',
 'event_count_explosions_m12',
 'event_count_explosions_w10',
 'event_count_explosions_w10_m12',
 'event_count_explosions_w5',
 'event_count_explosions_w5_m12',
 'event_count_violence',
 'event_count_violence_m12',
 'event_count_violence_w10',
 'event_count_violence_w10_m12',
 'event_count_violence_w5',
 'event_count_violence_w5_m12',
 'gini',
 'gini_m12',
 'nightlight_mean',
 'nightlight_mean_m12',
 'nightlight_std',
 'nightlight_std_m12',
 'sum_fatalities_battles',
 'sum_fatalities_battles_m12',
 'sum_fatalities_battles_w10',
 'sum_fatalities_battles_w10_m12',
 'sum_fatalities_battles_w5',
 'sum_fatalities_battles_w5_m12',
 'sum_fatalities_explosions',
 'sum_fatalities_explosions_w10',
 'sum_fatalities_explosions_w10_m12',
 'sum_fatalities_explosions_w5',
 'sum_fatalities_explosions_w5_m12',
 'sum_fatalities_violence',
 'sum_fatalities_violence_m12',
 'sum_fatalities_violence_w10',
 'sum_fatalities_violence_w10_m12',
 'sum_fatalities_violence_w5',
 'sum_fatalities_violence_w5_m12']

# --- Pre-merge nowcasting diff_set features into forecasting DataFrame ---
now_cols_to_merge = [c for c in diff_set if c in df_nowcasting.columns]
nowcasting_merge = df_nowcasting[['area_id', 'date'] + now_cols_to_merge].copy()
nowcasting_merge = nowcasting_merge.rename(columns={c: f'now_{c}' for c in now_cols_to_merge})
now_diff_set = [f'now_{c}' for c in now_cols_to_merge]
df = df.merge(nowcasting_merge, on=['area_id', 'date'], how='left')
print(f"Merged {len(now_diff_set)} nowcasting features. df shape: {df.shape}")

y_pred_test = pd.DataFrame()
date = "2022-01-01"
cutoff_date = "2023-01-01"

for i in range(2, 6):
    train_df = df[df['date'] < date].copy()
    test_df = df[(df['date'] >= date) & (df['date'] < cutoff_date)].copy()

    # Extract nowcasting features before dropping columns
    train_now_feats = train_df[now_diff_set]
    test_now_feats = test_df[now_diff_set]

    # Drop date, area_id, and now_* columns for Layer 1
    train_df = train_df.drop(['date', 'area_id'] + now_diff_set, axis=1)
    test_df = test_df.drop(['date', 'area_id'] + now_diff_set, axis=1)

    # Phase filtering
    train_df_new = train_df.drop([f'phase{j}_worse' for j in range(2, 6) if j != i], axis=1)
    test_df_new = test_df.drop([f'phase{j}_worse' for j in range(2, 6) if j != i], axis=1)
    train_df_new = train_df_new.dropna(subset=[f'phase{i}_worse'])
    test_df_new = test_df_new.dropna(subset=[f'phase{i}_worse'])
    test_index = test_df_new.index

    # LAYER 1
    X_train_L1 = train_df_new.drop(f'phase{i}_worse', axis=1)
    y_train = train_df_new[f'phase{i}_worse']
    X_test_L1 = test_df_new.drop(f'phase{i}_worse', axis=1)
    y_test = test_df_new[f'phase{i}_worse']

    layer_params = best_params_xgb_regressor_for_p3 if i == 3 else best_params_xgb_regressor
    model_L1 = xgb.XGBRegressor(**layer_params)
    model_L1.fit(X_train_L1, y_train)

    y_pred_L1 = pd.Series(model_L1.predict(X_test_L1), index=X_test_L1.index)
    train_residuals = pd.Series(
        y_train.to_numpy() - model_L1.predict(X_train_L1),
        index=X_train_L1.index
    )

    # LAYER 2 (index-aligned from same DataFrame)
    X_train_L2 = train_now_feats.loc[X_train_L1.index]
    X_test_L2 = test_now_feats.loc[X_test_L1.index]

    # Drop rows where ALL nowcasting features are NaN (XGBoost handles partial NaN natively)
    train_mask = X_train_L2.notna().any(axis=1)
    test_mask = X_test_L2.notna().any(axis=1)
    X_train_L2 = X_train_L2[train_mask]
    X_test_L2 = X_test_L2[test_mask]
    train_residuals_aligned = train_residuals.loc[X_train_L2.index]
    y_pred_L1_aligned = y_pred_L1.loc[X_test_L2.index]
    y_test_aligned = y_test.loc[X_test_L2.index]

    if X_train_L2.empty or X_test_L2.empty:
        print(f"Phase {i}: skipped (empty Layer 2 data)")
        continue

    model_L2 = xgb.XGBRegressor(**layer_params)
    model_L2.fit(X_train_L2, train_residuals_aligned)
    residual_pred = model_L2.predict(X_test_L2)
    y_pred_final = y_pred_L1_aligned.to_numpy() + residual_pred

    # Store in reference format
    row = {'y_pred': y_pred_final, 'y_test': y_test_aligned.to_numpy(), 'phase': [i]*len(y_pred_final)}
    if i == 5:
        row['test_index'] = X_test_L2.index
    y_pred_test = pd.concat([y_pred_test, pd.DataFrame(row)], ignore_index=True)

y_pred_test = convert_prob_to_phase(y_pred_test)
y_test = y_pred_test['overall_phase']
y_pred = y_pred_test['overall_phase_pred']
cm = confusion_matrix(y_test, y_pred)
accuracy_score_new, sensitivity, precision, overall_r2 = all_metrics(y_test, y_pred, cm, y_pred_test)
print("2022")
print("Accuracy:", accuracy_score_new)
print("Sensitivity:", sensitivity)
print("Precision:", precision)
print("Overall R²(Phase 3 or more):", overall_r2)


Merged 173 nowcasting features. df shape: (29621, 397)
2022
Accuracy: 0.5002407318247473
Sensitivity: 0.6024662360540223
Precision: 0.7328571428571429
Overall R²(Phase 3 or more): 0.3304985091280823


In [7]:
import os
out_dir = 'results/predictions/nowcasting'
os.makedirs(out_dir, exist_ok=True)

y_pred_test = y_pred_test.loc[:, (y_pred_test != 0).any(axis=0)]
df_extracted = df[['area_id', 'date']].copy()
df_extracted['test_index'] = df_extracted.index
y_pred_test = y_pred_test.merge(df_extracted, on='test_index', how='left')

PATH = r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\IPCCH_2017_2025_final_v12102025_with_zscores.csv'
df_ipcch = pd.read_csv(PATH)
df_ipcch = df_ipcch[['admin_code', 'lat', 'lon']].rename(columns={'admin_code': 'area_id'})
y_pred_test = y_pred_test.merge(df_ipcch, on='area_id', how='left')
y_pred_test.to_csv(os.path.join(out_dir, 'nowcasting_y_pred_test_2022.csv'), index=False)
%reset -f
